# What if Antarctic ice shelves become more damaged?

Ice shelves are floating extensions of the Antarctic ice sheet. They help hold back the flow of land ice into the ocean. Cracks, crevasses and rifts can weaken a shelf; if its support is reduced, land ice can flow faster and contribute to sea-level rise.

This notebook is part of the ice-shelf **what-if scenario** exploration. It lets you ask: *How might ice-shelf damage differ under alternative climate experiments, in a place and period I choose?* It explores existing projections; it does not run a new ice-sheet simulation.

## The research behind the explorer

[Izeboud et al. (2025), *Damage development on Antarctic ice shelves sensitive to climate warming*](https://www.nature.com/articles/s41558-025-02453-4), combined satellite observations with a machine-learning model linking damage to ice flow. Observations showed cycles of damage linked to changes in ice-shelf area, and a net decline in damaged area across Antarctica between 1997 and 2021. Much of that long-term decline was attributed to damaged ice breaking away as icebergs (calving), not necessarily to the shelves becoming healthier.

To explore the future, the researchers applied their damage model to projections from five ice-sheet models and four climate experiments. Faster flow, increased deformation and thinning were linked to more damage, especially under high-emissions scenarios. The historical findings and future projections answer different questions; one should not be read as a forecast of the other.

The study highlights a possible gap: if models do not adequately represent damage and its effects, they may underestimate ice-shelf retreat and Antarctic ice loss. That is a research concern, **not a prediction that a particular shelf will collapse**.

### How can damage and retreat be connected?

[View Figure 3 in the paper](https://www.nature.com/articles/s41558-025-02453-4/figures/3) (illustration credit: Iris van der Heide). The upper panels illustrate a recurring **damage cycle**: as a shelf advances, cracks can develop; calving can then remove damaged ice. Less damage remaining on a shelf does not necessarily mean less ice was lost.

The lower panels illustrate a possible **damage feedback**: if a shelf loses some of its ability to hold back land ice, faster flow and greater deformation can create more damage. Further calving could reduce that support still more. The study did not find evidence that this feedback caused runaway retreat during the observation period, and could not quantify its future strength. The damage estimates in this notebook do not by themselves tell us whether this feedback will occur.

### One result to keep in mind

For 2100, the paper reports an Antarctic-wide average damage change relative to control of about **−0.8%** under the lower-emissions pathway and **+3.2%** under the higher-emissions pathway. Individual models varied substantially, and sensitivity differed between regions. These numbers illustrate a range of possible responses, not a single forecast. This notebook's selected region, historical shelf mask and averaging choices differ from the paper's analysis, so its results need not reproduce those numbers.

## How to explore a what-if question

1. **Choose a place and years.** Pick an ice-shelf preset, draw a box or choose a point; shorter periods download faster.
2. **Fetch the data.** The notebook requests the selected region and years from the Digital Twin Ice Sheets API, or reuses an identical cached request.
3. **Compare maps.** Select an ice-sheet model and an experiment. See the experiment, its control and their difference for a chosen year.
4. **Explore time series.** Compare experiments or models over time, and choose whether the regional statistic uses the whole selected region or mapped ice shelves only.

## What do the scenarios and numbers mean?

An **experiment** is a precomputed combination of climate and ocean assumptions. The standard ISMIP6 experiments include a lower greenhouse-gas forcing pathway (RCP2.6), a higher one (RCP8.5), and different climate models. For example, when available, compare **exp05** and **exp07**: both use the NorESM1-M climate model but different pathways. The **ice-sheet model** is a separate choice: it describes how the ice sheet responds. Experiment descriptions are shown beside the selectors. These are alternative modelled futures, not probabilities or freely adjustable climate settings.

**Damage** is a modelled, dimensionless indicator related to features such as cracks and crevasses; zero indicates no damage. It is not a direct measurement of crack length, a probability of collapse or a sea-level-rise prediction. The study's model had difficulty reproducing the highest observed damage values, so relative changes are especially informative. Results depend on the models, scenarios and data available, and should be interpreted as exploratory projections rather than observations of the future.

Each experiment is compared with the **control run of the same ice-sheet model**: a reference simulation with climate forcing held constant from 2015 onward, not an observation of the future. A positive experiment-minus-control difference means more modelled damage than that control in the same year; it does not necessarily mean damage is increasing compared with an earlier year.

## 0. Setup

Run the setup cells first to install and load the tools used by the explorer. In Colab, the widget manager must be enabled for the interactive controls. Enter your DTC access token when prompted; it is not saved in the notebook.

In [ ]:
# Install what Colab does not ship with.
!pip install -q cartopy cmcrameri zarr
# >>> Adapt this line to however you normally install the Digital Twin query client <<<
!pip install -q dtc-query-client

In [ ]:
import sys

if "google.colab" in sys.modules:
    print("Installing dependencies...")
    %pip install -q xarray zarr netcdf4 cartopy fsspec aiohttp plotly seaborn ipyleaflet ipywidgets==7.7.1 \
        dtc-query-client jupyter-ui-poll
    # Ensure the newest version of the helpers package is installed
    %pip uninstall -y dtc_is_notebooks
    %pip install -q --upgrade --force-reinstall --no-deps --no-cache-dir  git+https://github.com/DTC-Ice-Sheets/dtc_is_notebooks.git
    # The interactive map (ipyleaflet) needs Colab's custom widget manager
    from google.colab import output

    output.enable_custom_widget_manager()
    print("Dependencies installed.")

In [ ]:
import asyncio
import getpass
import hashlib
import html
import json
import sys
import tempfile
import traceback
from contextlib import AbstractContextManager
from datetime import date
from pathlib import Path
from types import SimpleNamespace
from typing import Any, Mapping, Optional, Sequence, Tuple, Union

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shapely
import xarray as xr
from dtc_query_client import (
    ApiClient,
    Configuration,
    DataDownloadRequest,
    DatasetID,
    Epsg4326PolygonJson,
    GenericApi,
)
from dtc_query_client.helpers import wait_for_job
from dtc_query_client.models.job_status_response import JobStatusResponse
from IPython.display import display as ipy_display
from matplotlib.axes import Axes
from pyproj import Transformer
from shapely.geometry import Point, Polygon, box, mapping
from shapely.ops import transform as shp_transform

try:
    import cmcrameri.cm as cmc
    CMAP = cmc.batlow
except ImportError:
    CMAP = "inferno"

# ------------------------------------------------------------------ settings
HOST = "https://query.dtc-ice-sheets.org"
DARK = True          # dark map style (as in the original dashboard)
EPS = 1e-3           # percent change is masked where control damage <= EPS
GRID = 8000          # ISMIP6 grid spacing in m
LARGE_MB = 1500      # queries above this size need the "allow large queries" tick box
CACHE_DIR = Path("/content/data/damage_explorer") if "google.colab" in sys.modules else Path("data/damage_explorer")

# ------------------------------------------------------------------ authentication
# The hidden prompt keeps the access token out of the notebook.
TOKEN = getpass.getpass("DTC access token: ")
config = Configuration(host=HOST, access_token=TOKEN)

# ------------------------------------------------------------------ projection
# The data are on EPSG:3031 (true scale at 71°S). NB: ccrs.SouthPolarStereo() without
# true_scale_latitude is a *different* projection and would misalign coastlines slightly.
PROJ = ccrs.SouthPolarStereo(true_scale_latitude=-71)
LL2XY = Transformer.from_crs(4326, 3031, always_xy=True)
XY2LL = Transformer.from_crs(3031, 4326, always_xy=True)

# ------------------------------------------------------------------ shared state
STATE = SimpleNamespace(ds=None, mask=None, shelf_mask=None, region=None, point=None, mode=None,
                        radius_km=0, label="", limits={}, series={})
CACHE = {}       # exact query -> loaded dataset, avoids repeated disk reads
REFRESH = []     # callbacks that update the viewers after a successful fetch

## 1. Prepare the explorer

Run this cell to define the region, data and experiment helpers used below. You do not need to change it to compare scenarios.

**Technical note:** Regions are selected on the Antarctic polar-stereographic grid (EPSG:3031), converted to latitude and longitude for the API, then cropped and masked locally. A latitude-band query is used when the selection encloses the pole or crosses the antimeridian; the exact region is still applied locally.

In [ ]:
def _style() -> AbstractContextManager[None]:
    return plt.style.context("dark_background" if DARK else "default")

COAST = "white" if DARK else "black"

def _delta(exp: xr.DataArray, ctrl: xr.DataArray, mode: str) -> xr.DataArray:
    # change of an experiment relative to its control. mode: 'abs' or 'pct'
    if mode == "pct":
        return (exp - ctrl) / ctrl.where(ctrl > EPS) * 100
    return exp - ctrl

def _exp_label(ds: xr.Dataset, e: str) -> str:
    try:
        rcp = str(ds["rcp"].sel(experiment=e).item())
        gcm = str(ds["aogcm"].sel(experiment=e).item())
        return f"{e} · RCP{rcp} · {gcm}"
    except Exception:
        return str(e)

EXP_NOTES = {
    "exp01": "Low atmospheric change and medium-to-high ocean warming.",
    "exp02": "High atmospheric changes and medium ocean warming.",
    "exp03": "Low atmospheric change and medium-to-high ocean warming.",
    "exp04": "Large atmospheric warming and ocean warming that varies by region.",
    "exp05": "Low atmospheric change and medium-to-high ocean warming.",
    "exp06": "High atmospheric changes and medium ocean warming.",
    "exp07": "Low atmospheric change and medium-to-high ocean warming.",
    "exp08": "Large atmospheric warming and ocean warming that varies by region.",
    "exp09": "Tests ocean-forcing uncertainty using the 95th percentile of values.",
    "exp10": "Tests ocean-forcing uncertainty using the 5th percentile of values.",
    "exp11": "Includes ice-shelf hydrofracture (fracturing driven by surface meltwater).",
    "exp12": "Includes ice-shelf hydrofracture (fracturing driven by surface meltwater).",
    "exp13": "Tests ocean-forcing uncertainty using the PIGL calibration.",
}

def _experiment_notes(selected: Sequence[str]) -> str:
    if not selected:
        return "Select experiments to see their descriptions."
    items = [f"<li><b>{html.escape(e)}</b>: {html.escape(EXP_NOTES.get(e, 'No description available.'))}</li>"
             for e in selected]
    return "<b>Selected experiments</b><ul>" + "".join(items) + "</ul>"

ICE_SHELVES = cfeature.NaturalEarthFeature(
    "physical", "antarctic_ice_shelves_polys", "10m", facecolor="none")

# ------------------------------------------------------------------ region helpers
def lonlat_box_to_xy_km(lon0: float, lon1: float, lat0: float, lat1: float,
                         n: int = 60) -> Tuple[float, float, float, float]:
    """Return the projected, grid-aligned bounds of a longitude/latitude box in kilometres."""
    # bounding box (km, snapped to the 8 km grid) in EPSG:3031 of a lon/lat box
    lons = np.r_[np.linspace(lon0, lon1, n), np.full(n, lon1), np.linspace(lon1, lon0, n), np.full(n, lon0)]
    lats = np.r_[np.full(n, lat0), np.linspace(lat0, lat1, n), np.full(n, lat1), np.linspace(lat1, lat0, n)]
    x, y = LL2XY.transform(lons, lats)
    return (float(np.floor(x.min() / GRID) * GRID / 1000),
            float(np.ceil(x.max() / GRID) * GRID / 1000),
            float(np.floor(y.min() / GRID) * GRID / 1000),
            float(np.ceil(y.max() / GRID) * GRID / 1000))

def to_query_polygon(region_xy: Polygon, pad_m: float) -> Polygon:
    """Convert a padded projected region to a geographic polygon for the API."""
    # EPSG:3031 region -> EPSG:4326 polygon for the API (padded so no edge cells are lost)
    padded = region_xy.buffer(pad_m, join_style="mitre")
    ll = shp_transform(XY2LL.transform, padded.segmentize(5000))
    lons, lats = np.array(ll.exterior.coords).T
    if padded.contains(Point(0, 0)) or np.ptp(lons) > 180:
        # encloses the pole / crosses the antimeridian -> latitude band
        return box(-180, -90, 180, min(lats.max() + 1, -50))
    return ll

def estimate_size_mb(region_xy: Polygon, n_years: int, n_series: int = 25) -> float:
    """Estimate the size in megabytes of a query result."""
    x0, y0, x1, y1 = region_xy.bounds
    cells = ((x1 - x0) / GRID + 1) * ((y1 - y0) / GRID + 1)
    return cells * n_years * n_series * 4 / 1e6

def _fmt_mb(mb: float) -> str:
    return f"{mb/1000:.1f} GB" if mb >= 1000 else f"{mb:.0f} MB"

def build_mask(ds: xr.Dataset, region: Polygon, point: Optional[Tuple[float, float]],
               radius_m: float) -> xr.DataArray:
    """Select grid-cell centres inside the region or nearest to a point."""
    grid_x, grid_y = np.meshgrid(ds.x.values, ds.y.values)
    if point is not None and radius_m == 0:          # single nearest cell
        d = (grid_x - point[0]) ** 2 + (grid_y - point[1]) ** 2
        m = d == d.min()
    else:
        m = shapely.intersects_xy(region, grid_x, grid_y)
    if not m.any():
        raise ValueError("No grid cells fall inside the selected region.")
    return xr.DataArray(m, coords={"y": ds.y, "x": ds.x}, dims=("y", "x"))

# ------------------------------------------------------------------ API access
async def _submit(poly_ll: Polygon, y0: int, y1: int) -> JobStatusResponse:
    async with ApiClient(config) as client:
        submission = await GenericApi(client).download_data(
            DataDownloadRequest(
                dataset_id=DatasetID.ISMIP6_DAMAGE,
                start_time=date(y0, 1, 1),
                end_time=date(y1, 12, 31),
                epsg4326_polygon_json=Epsg4326PolygonJson.from_dict(mapping(poly_ll)),
            )
        )
        set_status(f"Job {submission.job_id} submitted, waiting for the result ...")
        return await wait_for_job(client, submission.job_id, poll_interval=2.0)

def _open_and_load(path: Union[str, Path]) -> xr.Dataset:
    # pull everything into memory once so the viewers are instant
    # (for remote stores you may need fsspec/s3fs/gcsfs installed)
    with xr.open_dataset(path, engine="zarr") as ds:
        return ds.load()

def _cache_path(key: Tuple[str, str, str, int, str, int, int, int]) -> Path:
    digest = hashlib.sha256(json.dumps(key, separators=(",", ":")).encode("utf-8")).hexdigest()
    return CACHE_DIR / f"{digest}.zarr"

def _save_cached_dataset(ds: xr.Dataset, path: Path) -> None:
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix="damage-cache-", dir=CACHE_DIR) as tmp:
        staging = Path(tmp) / "data.zarr"
        cached = ds.drop_encoding()
        for variable in cached.variables.values():
            if np.issubdtype(variable.dtype, np.datetime64):
                for key in ("units", "calendar"):
                    if key in variable.attrs:
                        variable.encoding[key] = variable.attrs.pop(key)
        cached.to_zarr(staging, mode="w")
        staging.rename(path)

def _prepare(ds: xr.Dataset, region: Polygon, margin_m: float) -> xr.Dataset:
    # time -> year dimension; crop to the region bounding box (+ margin)
    ds = ds.assign_coords(year=("time", ds["time"].dt.year.values)).swap_dims({"time": "year"})
    x0, y0, x1, y1 = region.bounds
    ix = np.where((ds.x.values >= x0 - margin_m - 1) & (ds.x.values <= x1 + margin_m + 1))[0]
    iy = np.where((ds.y.values >= y0 - margin_m - 1) & (ds.y.values <= y1 + margin_m + 1))[0]
    if ix.size == 0 or iy.size == 0:
        raise ValueError("The returned data do not overlap the selected region.")
    return ds.isel(x=ix, y=iy)

## 2. Select region and time span, then fetch

Pick a place and time period, then click **Fetch data**. The map previews your selection; it does not yet show damage.

* **Box**: choose a shelf preset or adjust the x/y ranges. Presets are approximate rectangular starting points, not exact shelf boundaries.
* **Point**: enter longitude and latitude. Radius 0 uses the nearest 8 km grid cell; a larger radius selects a square around the point.
* Fewer years means a smaller download. The whole continent is only practical for a few years. The progress bar shows fetch stages, not download percentage.
* Exact repeat queries reuse a local Zarr cache in `/content/data/damage_explorer` on Colab (or `data/damage_explorer` elsewhere). Colab's local storage is temporary and is lost when the runtime is deleted; set `CACHE_DIR` in setup to a mounted Drive folder for longer-term storage. Cached data do not automatically refresh when the source dataset changes; delete the relevant cache folder and restart the runtime (or clear `CACHE`) to download again.

Ice-shelf outlines and the named shelf presets use [Natural Earth Antarctic ice shelves](https://www.naturalearthdata.com/downloads/10m-physical-vectors/10m-antarctic-ice-shelves/), based on imagery from 2003–2004. Outlines are geographic context, not the model's shelf mask, and may differ from present-day or projected boundaries. Cartopy downloads the outlines on first use.

In [ ]:
PRESETS = {
    "— custom —": None,
    "Whole Antarctica (one year)": {"xy_km": (-3100, 3100, -3100, 3100), "years": (2100, 2100)},
    "Amundsen Sea (your example box)": {"lonlat": (-110, -95, -76, -73)},
    "Larsen C (approx.)": {"lonlat": (-67, -58, -69.5, -66)},
    "Ronne–Filchner (approx.)": {"lonlat": (-85, -25, -84, -74)},
    "Amery (approx.)": {"lonlat": (65, 80, -74, -68)},
    "Ross (approx.)": {"xy_km": (-650, 450, -1350, -500)},
    "Abbott Ice Shelf": {"xy_km": (-1984, -1784, -480, 56)},
    "Amery Ice Shelf": {"xy_km": (1656, 2272, 584, 864)},
    "Cook Ice Shelf": {"xy_km": (1000, 1168, -2160, -2008)},
    "Dibble Iceberg Tongue": {"xy_km": (1808, 1912, -1904, -1776)},
    "Dotson Ice Shelf": {"xy_km": (-1624, -1464, -728, -584)},
    "Ekström Ice Shelf": {"xy_km": (-392, -240, 1952, 2144)},
    "Filchner Ice Shelf": {"xy_km": (-928, -392, 536, 1064)},
    "Fimbul Ice Shelf": {"xy_km": (-136, 144, 1968, 2272)},
    "Getz Ice Shelf": {"xy_km": (-1608, -1152, -1208, -688)},
    "Jelbart Ice Shelf": {"xy_km": (-256, -72, 1992, 2176)},
    "Larsen B Ice Shelf": {"xy_km": (-2456, -2336, 1296, 1408)},
    "Larsen Ice Shelf": {"xy_km": (-2384, -1584, 880, 1320)},
    "Lazarev Ice Shelf": {"xy_km": (464, 632, 2040, 2208)},
    "Mertz Glacier Tongue": {"xy_km": (1368, 1496, -2168, -1936)},
    "Moscow University Ice Shelf": {"xy_km": (2120, 2304, -1408, -1112)},
    "Nimis Glacier Tongue": {"xy_km": (1208, 1352, -2064, -1960)},
    "Rennick Glacier": {"xy_km": (600, 728, -2096, -1944)},
    "Riiser-Larsen Ice Shelf": {"xy_km": (-784, -344, 1328, 1992)},
    "Ronne Ice Shelf": {"xy_km": (-1536, -584, 104, 936)},
    "Ross Ice Shelf": {"xy_km": (-624, 424, -1376, -448)},
    "Shackleton Ice Shelf": {"xy_km": (2480, 2776, -608, -192)},
    "Stange Ice Shelf": {"xy_km": (-1928, -1688, 320, 504)},
    "Sulzberger Ice Shelf": {"xy_km": (-848, -616, -1312, -1096)},
    "Thwaites Glacier Tongue": {"xy_km": (-1632, -1496, -536, -352)},
    "Totten Glacier": {"xy_km": (2224, 2352, -1184, -968)},
    "Venable Ice Shelf": {"xy_km": (-1904, -1784, 16, 168)},
    "Voyeykov Ice Shelf": {"xy_km": (1968, 2104, -1592, -1488)},
    "West Ice Shelf": {"xy_km": (2384, 2632, 24, 400)},
    "Wilkins Ice Shelf": {"xy_km": (-2152, -1912, 544, 776)},
}

_st = {"description_width": "90px"}
_wide = widgets.Layout(width="700px")
w_preset = widgets.Dropdown(options=list(PRESETS), description="Preset:", style=_st)
w_mode = widgets.ToggleButtons(options=["Box", "Point"], description="Region:", style=_st)
w_x = widgets.FloatRangeSlider(value=[-1848, -1440], min=-3100, max=3100, step=8, description="x (km)",
                               continuous_update=False, readout_format=".0f", layout=_wide, style=_st)
w_y = widgets.FloatRangeSlider(value=[-632, -136], min=-3100, max=3100, step=8, description="y (km)",
                               continuous_update=False, readout_format=".0f", layout=_wide, style=_st)
w_lon = widgets.BoundedFloatText(value=-100.0, min=-180, max=180, step=0.5, description="lon (°E)", style=_st)
w_lat = widgets.BoundedFloatText(value=-75.0, min=-90, max=-50, step=0.5, description="lat (°N)", style=_st)
w_rad = widgets.IntSlider(value=0, min=0, max=80, step=8, description="radius (km)",
                          continuous_update=False, style=_st)
w_years = widgets.IntRangeSlider(value=[2015, 2100], min=2015, max=2100, description="years",
                                 continuous_update=False, layout=_wide, style=_st)
w_allow = widgets.Checkbox(value=False, description="allow large queries", indent=False)
w_fetch = widgets.Button(description="Fetch data", button_style="primary", icon="download")
w_info = widgets.HTML()
w_progress = widgets.IntProgress(value=0, min=0, max=10, description="Stage:", bar_style="info",
                                 layout=widgets.Layout(width="400px"))
status = widgets.HTML(value="<b>Status:</b> Ready to fetch data.")

def set_status(msg: str, kind: str = "info") -> None:
    """Update the fetch status shown in the notebook."""
    colour = {"info": "inherit", "ok": "#2e9e4f", "warn": "#b36b00", "error": "#d64545"}[kind]
    status.value = f"<b>Status:</b> <span style='color:{colour}'>{msg}</span>"

def current_region() -> Tuple[Polygon, Optional[Tuple[float, float]]]:
    """Return the selected projected region and optional point."""
    if w_mode.value == "Box":
        (x0, x1), (y0, y1) = w_x.value, w_y.value
        return box(x0 * 1e3, y0 * 1e3, x1 * 1e3, y1 * 1e3), None
    px, py = LL2XY.transform(w_lon.value, w_lat.value)
    h = max(w_rad.value * 1e3, GRID / 2)
    return box(px - h, py - h, px + h, py + h), (px, py)

def _clear_status_and_progress(_: object = None) -> None:
    if w_fetch.disabled:
        return
    set_status("Ready to fetch data.")
    w_progress.bar_style = "info"
    w_progress.value = 0

# Dynamic UI clearing observers linked to variables
for w in [w_mode, w_x, w_y, w_lon, w_lat, w_rad, w_years]:
    w.observe(_clear_status_and_progress, "value")

def _apply_preset(_: object) -> None:
    p = PRESETS[w_preset.value]
    if p is None:
        return
    x0, x1, y0, y1 = lonlat_box_to_xy_km(*p["lonlat"]) if "lonlat" in p else p["xy_km"]
    w_mode.value = "Box"
    w_x.value = (max(x0, -3100), min(x1, 3100))
    w_y.value = (max(y0, -3100), min(y1, 3100))
    if "years" in p:
        w_years.value = p["years"]
    _clear_status_and_progress()
w_preset.observe(_apply_preset, "value")

def draw_preview(mode: object, x: object, y: object, lon: object, lat: object, rad: object,
                 years: Tuple[int, int]) -> None:
    """Draw the selected region and update its estimated download size."""
    region, point = current_region()
    n_years = years[1] - years[0] + 1
    mb = estimate_size_mb(region, n_years)
    colour = "#d98e04" if mb > LARGE_MB else "#888"
    w_info.value = (f"<span style='color:{colour}'>Estimated download ≈ <b>{_fmt_mb(mb)}</b> "
                    f"({n_years} years × 25 model/experiment combinations)</span>")
    with _style():
        fig = plt.figure(figsize=(5, 5))
        ax = fig.add_subplot(projection=PROJ)
        ax.set_extent([-3.1e6, 3.1e6, -3.1e6, 3.1e6], crs=PROJ)
        ax.coastlines("50m", color=COAST, linewidth=0.6)
        ax.add_feature(ICE_SHELVES, edgecolor="teal", linewidth=0.6, zorder=3)
        ax.gridlines(linewidth=0.3, alpha=0.5)
        xs, ys = region.exterior.xy
        ax.fill(xs, ys, transform=PROJ, color="tab:red", alpha=0.4)
        ax.plot(xs, ys, transform=PROJ, color="tab:red", lw=1.2)
        if point is not None:
            ax.plot(*point, "x", transform=PROJ, color="magenta", ms=9, mew=2)
        ax.set_title("Selected region (EPSG:3031)", fontsize=10)
        plt.show()

preview_out = widgets.interactive_output(
    draw_preview, {"mode": w_mode, "x": w_x, "y": w_y, "lon": w_lon, "lat": w_lat, "rad": w_rad, "years": w_years})

box_ui = widgets.VBox([w_x, w_y])
point_ui = widgets.VBox([widgets.HBox([w_lon, w_lat]), w_rad])
def _toggle(_: object = None) -> None:
    box_ui.layout.display = "" if w_mode.value == "Box" else "none"
    point_ui.layout.display = "" if w_mode.value == "Point" else "none"
w_mode.observe(_toggle, "value")
_toggle()

async def _submit_with_progress(poly_ll: Polygon, y0: int, y1: int) -> JobStatusResponse:
    async with ApiClient(config) as client:
        submission = await GenericApi(client).download_data(
            DataDownloadRequest(
                dataset_id=DatasetID.ISMIP6_DAMAGE,
                start_time=date(y0, 1, 1),
                end_time=date(y1, 12, 31),
                epsg4326_polygon_json=Epsg4326PolygonJson.from_dict(mapping(poly_ll)),
            )
        )

        w_progress.value = 2
        set_status(f"Job {submission.job_id} submitted. Polling for results...")
        await asyncio.sleep(0.05) # Force-yield thread so UI renders progress bar instantly

        # Use wait_for_job helper which is imported from dtc_query_client.helpers
        job_result = await wait_for_job(client, submission.job_id, poll_interval=2.0)
        await asyncio.sleep(0.05)
        return job_result

async def _fetch_and_store() -> None:
    w_fetch.disabled = True
    w_progress.bar_style = "info"
    w_progress.value = 0
    set_status("Preparing query...")
    await asyncio.sleep(0.05) # Yield to refresh display instantly
    try:
        region, point = current_region()
        y0, y1 = w_years.value
        radius_km, lon, lat = w_rad.value, w_lon.value, w_lat.value
        x_bounds, y_bounds = w_x.value, w_y.value
        mb = estimate_size_mb(region, y1 - y0 + 1)
        if mb > LARGE_MB and not w_allow.value:
            set_status(f"Estimated size ≈ {_fmt_mb(mb)}. Tick 'allow large queries' or reduce region / years.", "warn")
            w_progress.bar_style = "warning"
            return
        is_point = point is not None
        margin = 5 * GRID if is_point else 0
        key = ("v1", HOST, str(DatasetID.ISMIP6_DAMAGE), GRID, region.wkt, int(y0), int(y1), margin)
        cache_path = _cache_path(key)
        if key in CACHE:
            ds_raw = CACHE[key]
            source = "memory cache"
            set_status("Using in-memory cached result ...")
            w_progress.value = 8
            await asyncio.sleep(0.05)
        elif cache_path.exists():
            set_status("Loading locally cached result ...")
            w_progress.value = 5
            await asyncio.sleep(0.05)
            ds_raw = await asyncio.to_thread(_open_and_load, cache_path)
            CACHE[key] = ds_raw
            source = "local cache"
            w_progress.value = 8
            await asyncio.sleep(0.05)
        else:
            set_status("Submitting query...")
            w_progress.value = 1
            await asyncio.sleep(0.05)
            poly_ll = to_query_polygon(region, pad_m=margin + 2 * GRID)
            result = await _submit_with_progress(poly_ll, y0, y1)
            set_status("API job completed! Downloading and loading data into memory...")
            w_progress.value = 4
            await asyncio.sleep(0.05)
            path = result.outputs["main"]["output_path"]
            ds_raw = await asyncio.to_thread(_open_and_load, path)
            set_status("Saving downloaded dataset to local cache...")
            w_progress.value = 6
            await asyncio.sleep(0.05)
            await asyncio.to_thread(_save_cached_dataset, ds_raw, cache_path)
            CACHE[key] = ds_raw
            source = "API; saved to local cache"
        set_status("Data loaded! Preparing dataset and local crop...")
        w_progress.value = 9
        await asyncio.sleep(0.05)
        ds = _prepare(ds_raw, region, margin)
        STATE.ds, STATE.region, STATE.point, STATE.mode = ds, region, point, ("Point" if is_point else "Box")
        STATE.radius_km = radius_km
        STATE.mask = build_mask(ds, region, point, radius_km * 1e3)
        STATE.shelf_mask = None
        STATE.limits, STATE.series = {}, {}
        if is_point:
            STATE.label = (f"point lon {lon:.2f}, lat {lat:.2f} "
                           f"(x={point[0]/1e3:.0f}, y={point[1]/1e3:.0f} km), radius {radius_km} km")
        else:
            (x0_, x1_), (y0_, y1_) = x_bounds, y_bounds
            STATE.label = f"box x[{x0_:.0f}, {x1_:.0f}] km, y[{y0_:.0f}, {y1_:.0f}] km"
        for cb in REFRESH:
            cb()
        globals()["ds"] = ds
        w_progress.value = 10
        w_progress.bar_style = "success"
        set_status(f"Loaded from {source}: {dict(ds.sizes)}. Use the map / time-series viewers below.", "ok")
    except Exception as e:
        w_progress.bar_style = "danger"
        set_status(f"<b>{type(e).__name__}</b>: {html.escape(str(e))}"
                   f"<pre style='font-size:11px'>{html.escape(traceback.format_exc())}</pre>", "error")
    finally:
        w_fetch.disabled = False

def _on_fetch(_: object) -> None:
    if w_fetch.disabled:
        return
    w_fetch.disabled = True
    w_progress.bar_style = "info"
    w_progress.value = 0
    set_status("Starting query...")
    asyncio.ensure_future(_fetch_and_store())
w_fetch.on_click(_on_fetch)

ipy_display(widgets.VBox([w_preset, w_mode, box_ui, point_ui, w_years,
                      widgets.HBox([w_fetch, w_allow]), status, w_progress, w_info, preview_out]))

## 3. Maps: experiment vs. control and the change

Choose a model, experiment and year to see three views of the same area:

1. **Experiment:** projected damage under the chosen what-if scenario.
2. **Control:** damage in the same model's reference run.
3. **Change:** experiment minus control. Positive values indicate more damage than the control; negative values indicate less.

The first two panels share a colour scale. Choose an absolute or percent difference; percent changes are hidden where control damage is too close to zero to make the ratio meaningful. Tick **fixed colour scale** to compare colours consistently while stepping through years. Teal lines show the historical ice-shelf outlines described above.

In [ ]:
m_model = widgets.Dropdown(options=[], description="Model:", style={"description_width": "90px"})
m_exp = widgets.Dropdown(options=[], description="Experiment:", style={"description_width": "90px"},
                         layout=widgets.Layout(width="480px"))
m_year = widgets.SelectionSlider(options=[2015], description="Year:", continuous_update=False,
                                 style={"description_width": "90px"}, layout=widgets.Layout(width="700px"))
m_play = widgets.Play(value=0, min=0, max=0, step=1, interval=600)
m_diff = widgets.Dropdown(options=[("Absolute change (exp − ctrl)", "abs"), ("Percent change vs. ctrl", "pct")],
                          description="Change:", style={"description_width": "90px"})
m_fixed = widgets.Checkbox(value=False, description="fixed colour scale over all years", indent=False)
m_exp_notes = widgets.HTML()

def _show_map_exp_note(change: Mapping[str, Any]) -> None:
    m_exp_notes.value = _experiment_notes((change["new"],) if change["new"] else ())
m_exp.observe(_show_map_exp_note, names="value")
_show_map_exp_note({"new": m_exp.value})

def _play_to_slider(ch: Mapping[str, Any]) -> None:
    if 0 <= ch["new"] < len(m_year.options):
        m_year.index = int(ch["new"])
m_play.observe(_play_to_slider, "value")

def _refresh_maps() -> None:
    ds = STATE.ds
    if ds is not None:
        m_model.options = [str(m) for m in ds.model.values]
        m_exp.options = [(_exp_label(ds, e), str(e)) for e in ds.experiment.values if str(e) != "ctrl"]
        yrs = [int(v) for v in ds.year.values]
        m_year.options = yrs
        m_year.index = int(len(yrs) - 1)
        m_play.max = int(len(yrs) - 1)
        m_play.value = int(len(yrs) - 1)
REFRESH.append(_refresh_maps)

def _limits(exp: xr.DataArray, ctrl: xr.DataArray, diff: xr.DataArray) -> Tuple[float, float, float]:
    both = np.concatenate([exp.values.ravel(), ctrl.values.ravel()])
    both = both[np.isfinite(both)]
    vmin, vmax = np.percentile(both, [1, 99]) if both.size else (0.0, 1.0)
    d = np.abs(diff.values[np.isfinite(diff.values)])
    dlim = np.percentile(d, 98) if d.size else 1.0
    return vmin, max(vmax, vmin + 1e-6), max(dlim, 1e-6)

def _fixed_limits(model: str, experiment: str, diff_mode: str) -> Tuple[float, float, float]:
    key = (model, experiment, diff_mode)
    if key not in STATE.limits:
        damage = STATE.ds["damage"].sel(model=model)
        s = max(1, damage.sizes["y"] // 150)
        sub = damage.isel(year=slice(None, None, 4), y=slice(None, None, s), x=slice(None, None, s))
        e, c = sub.sel(experiment=experiment), sub.sel(experiment="ctrl")
        STATE.limits[key] = _limits(e, c, _delta(e, c, diff_mode))
    return STATE.limits[key]

def _overlay_region(ax: Axes) -> None:
    if STATE.mode == "Point":
        xs, ys = STATE.region.exterior.xy
        if STATE.radius_km > 0:
            ax.plot(xs, ys, transform=PROJ, color="magenta", lw=1.2, ls="--")
        ax.plot(*STATE.point, "x", transform=PROJ, color="magenta", ms=8, mew=1.8)

def plot_maps(model: Optional[str], experiment: Optional[str], year: Optional[int],
              diff_mode: str, fixed: bool) -> None:
    """Plot experiment, control and difference maps for the selected year."""
    if STATE.ds is None:
        print("Fetch data first (step 2).")
        return
    if not model or not experiment or not year:
        return
    damage = STATE.ds["damage"].sel(model=model)
    exp = damage.sel(experiment=experiment, year=int(year))
    ctrl = damage.sel(experiment="ctrl", year=int(year))
    diff = _delta(exp, ctrl, diff_mode)
    vmin, vmax, dlim = _fixed_limits(model, experiment, diff_mode) if fixed else _limits(exp, ctrl, diff)

    x, y = STATE.ds.x.values, STATE.ds.y.values
    ext = (x.min() - GRID / 2, x.max() + GRID / 2, y.min() - GRID / 2, y.max() + GRID / 2)
    pw = 5.5
    ph = float(np.clip(pw * (ext[3] - ext[2]) / (ext[1] - ext[0]), 2.5, 8))
    coast_res = "50m" if (ext[1] - ext[0]) > 4e6 else "10m"
    dlabel = "Δ damage (exp − ctrl)" if diff_mode == "abs" else "change vs. ctrl (%)"

    with _style():
        fig, axes = plt.subplots(1, 3, figsize=(3 * pw + 2, ph + 1.3),
                                 subplot_kw={"projection": PROJ}, constrained_layout=True)
        panels = [(exp, CMAP, vmin, vmax, f"{model} · {experiment} · {year}"),
                  (ctrl, CMAP, vmin, vmax, f"{model} · ctrl · {year}"),
                  (diff, "RdBu_r", -dlim, dlim, dlabel)]
        ims = []
        for ax, (da, cm, lo, hi, title) in zip(axes, panels):
            ims.append(ax.pcolormesh(x, y, da.values, transform=PROJ, cmap=cm, vmin=lo, vmax=hi, shading="auto"))
            ax.set_extent(ext, crs=PROJ)
            ax.coastlines(coast_res, color=COAST, linewidth=0.7)
            ax.add_feature(ICE_SHELVES, edgecolor="teal", linewidth=0.7, zorder=3)
            ax.set_title(title, fontsize=10)
            _overlay_region(ax)
        fig.colorbar(ims[0], ax=axes[:2], shrink=0.8, label="damage")
        fig.colorbar(ims[2], ax=axes[2], shrink=0.8, label=dlabel)
        plt.show()

# Automatically initialize options if data has already been fetched
if STATE.ds is not None:
    _refresh_maps()

maps_out = widgets.interactive_output(
    plot_maps, {"model": m_model, "experiment": m_exp, "year": m_year, "diff_mode": m_diff, "fixed": m_fixed})
ipy_display(widgets.VBox([widgets.HBox([m_model, m_exp]), m_exp_notes, widgets.HBox([m_year, m_play]),
                      widgets.HBox([m_diff, m_fixed]), maps_out]))

## 4. Time series over the selected region or point

Ask how the selected scenarios compare over time. Group columns by **model** to compare experiments, or by **experiment** to compare models. Switch between light and dark figure styles as needed.

**Top row:** damage for the experiment (solid) and its control (dashed). **Bottom row:** the difference relative to the control of the same model. Choose the mean, median or maximum to summarise the selected area. The optional ensemble is the average across selected models; it appears as a separate column when grouped by model, or as an additional line when grouped by experiment.

Choose the whole selected region or only grid-cell centres inside the historical Natural Earth ice-shelf polygons. This is a fixed geographical approximation, not the model's evolving ice-shelf mask; cells outside the selected region are always excluded. Missing damage values are excluded from the statistic. For percent change, the ratio is taken of the region statistic (not averaged per cell). For the ensemble, damage and control are first averaged across models and then compared; this differs from the paper's average of individual-model percentage changes.

Descriptions beneath the experiment selector are adapted from the notes in [Table 2 of the ISMIP6 study](https://tc.copernicus.org/articles/14/2331/2020/). Use Ctrl (Windows/Linux) or Cmd (Mac) to select multiple entries.

In [ ]:
t_models = widgets.SelectMultiple(options=[], description="Models:", rows=5, style={"description_width": "70px"})
t_exps = widgets.SelectMultiple(options=[], description="Experiments:", rows=5,
                                style={"description_width": "90px"}, layout=widgets.Layout(width="520px"))
t_stat = widgets.Dropdown(options=["mean", "median", "max"], description="Statistic:",
                          style={"description_width": "90px"})
t_diff = widgets.Dropdown(options=[("Absolute change (exp − ctrl)", "abs"), ("Percent change vs. ctrl", "pct")],
                          description="Change:", style={"description_width": "90px"})
t_ens = widgets.Checkbox(value=False, description="add ensemble mean", indent=False)
t_theme = widgets.ToggleButtons(options=[("Dark", "dark"), ("Light", "light")],
                                value="dark" if DARK else "light", description="Figure:")
t_group = widgets.ToggleButtons(options=[("Model", "model"), ("Experiment", "experiment")],
                                value="model", description="Group by:")
t_area = widgets.ToggleButtons(options=[("Whole region", "domain"), ("Ice shelves", "shelves")],
                               value="domain", description="Statistic:")
EXP_COLORS = {}
MODEL_COLORS = {}
t_exp_notes = widgets.HTML()

def _show_exp_notes(change: Mapping[str, Any]) -> None:
    t_exp_notes.value = _experiment_notes(change["new"])

t_exps.observe(_show_exp_notes, names="value")
_show_exp_notes({"new": t_exps.value})

def _refresh_ts() -> None:
    ds = STATE.ds
    models = [str(m) for m in ds.model.values]
    exps = [str(e) for e in ds.experiment.values if str(e) != "ctrl"]
    EXP_COLORS.clear()
    EXP_COLORS.update({e: plt.cm.tab10(i) for i, e in enumerate(exps)})
    MODEL_COLORS.clear()
    MODEL_COLORS.update({m: plt.cm.tab10(i) for i, m in enumerate(models)})
    t_models.options = models
    t_exps.options = [(_exp_label(ds, e), e) for e in exps]
    t_models.value = (models[0],)
    t_exps.value = tuple(exps)
REFRESH.append(_refresh_ts)
if STATE.ds is not None:
    _refresh_ts()

def _shelf_mask() -> xr.DataArray:
    if getattr(STATE, "shelf_mask", None) is None:
        shelves = shapely.union_all([shp_transform(LL2XY.transform, geom) for geom in ICE_SHELVES.geometries()])
        grid_x, grid_y = np.meshgrid(STATE.ds.x.values, STATE.ds.y.values)
        mask = shapely.intersects_xy(shelves, grid_x, grid_y)
        STATE.shelf_mask = xr.DataArray(mask, coords={"y": STATE.ds.y, "x": STATE.ds.x}, dims=("y", "x"))
    return STATE.shelf_mask

def _series(stat: str, area: str = "domain") -> xr.DataArray:
    if area not in ("domain", "shelves"):
        raise ValueError(f"Unknown statistic area: {area}")
    key = (stat, area)
    if key not in STATE.series:
        mask = STATE.mask if area == "domain" else STATE.mask & _shelf_mask()
        if not bool(mask.any()):
            raise ValueError("No grid-cell centres in the selected region fall on a mapped ice shelf.")
        damage = STATE.ds["damage"].where(mask)
        STATE.series[key] = getattr(damage, stat)(("y", "x"))
    return STATE.series[key]

def plot_timeseries(models: Sequence[str], exps: Sequence[str], stat: str, diff_mode: str,
                    ens: bool, theme: str, group: str, area: str) -> None:
    """Plot regional damage and control differences over time."""
    if STATE.ds is None:
        print("Fetch data first (step 2).")
        return
    models, exps = list(models), list(exps)
    if not models or not exps:
        print("Select at least one model and one experiment.")
        return
    if area == "shelves" and not bool((STATE.mask & _shelf_mask()).any()):
        print("No grid-cell centres in the selected region fall on a mapped ice shelf.")
        return
    series = _series(stat, area)
    if area == "shelves" and not bool(series.sel(model=models, experiment=["ctrl"] + exps).notnull().any()):
        print("No damage values are available on mapped ice shelves for the selected models and experiments.")
        return
    years = series.year.values
    add_ensemble = ens and len(models) > 1
    cols = (models + (["Ensemble mean"] if add_ensemble else [])) if group == "model" else exps
    dlabel = "Δ damage (exp − ctrl)" if diff_mode == "abs" else "change vs. ctrl (%)"

    with plt.style.context("dark_background" if theme == "dark" else "default"):
        fig, axes = plt.subplots(2, len(cols), figsize=(4.4 * len(cols) + 1.5, 7.5), sharex=True, sharey="row",
                                 squeeze=False, constrained_layout=True, gridspec_kw={"height_ratios": [1, 1.2]})
        for j, name in enumerate(cols):
            a0, a1 = axes[0, j], axes[1, j]
            if group == "model":
                sel = series.sel(model=models).mean("model") if name == "Ensemble mean" else series.sel(model=name)
                ctrl = sel.sel(experiment="ctrl")
                a0.plot(years, ctrl, "--", color="0.65", lw=1.6, label="ctrl")
                for e in exps:
                    s = sel.sel(experiment=e)
                    a0.plot(years, s, color=EXP_COLORS[e], lw=1.8, label=e)
                    a1.plot(years, _delta(s, ctrl, diff_mode), color=EXP_COLORS[e], lw=1.8, label=e)
            else:
                for model in models + (["Ensemble mean"] if add_ensemble else []):
                    sel = (series.sel(model=models).mean("model") if model == "Ensemble mean"
                           else series.sel(model=model))
                    ctrl, s = sel.sel(experiment="ctrl"), sel.sel(experiment=name)
                    color = (("white" if theme == "dark" else "black")
                             if model == "Ensemble mean" else MODEL_COLORS[model])
                    a0.plot(years, ctrl, "--", color=color, lw=1.3, label=f"{model} ctrl")
                    a0.plot(years, s, color=color, lw=1.8, label=model)
                    a1.plot(years, _delta(s, ctrl, diff_mode), color=color, lw=1.8, label=model)
            a1.axhline(0, color="0.5", lw=0.8)
            a0.set_title(name, fontsize=10)
            a1.set_xlabel("year")
            for a in (a0, a1):
                a.grid(alpha=0.25)
        axes[0, 0].set_ylabel(f"damage ({stat})")
        axes[1, 0].set_ylabel(dlabel)
        axes[0, -1].legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1.0))
        fig.suptitle(f"Region: {STATE.label} · {'ice shelves' if area == 'shelves' else 'whole region'}", fontsize=10)
        plt.show()

ts_out = widgets.interactive_output(
    plot_timeseries, {"models": t_models, "exps": t_exps, "stat": t_stat, "diff_mode": t_diff, "ens": t_ens,
                      "theme": t_theme, "group": t_group, "area": t_area})
ipy_display(widgets.VBox([widgets.HBox([t_models, t_exps]), t_exp_notes,
                      widgets.HBox([t_stat, t_diff, t_ens]), widgets.HBox([t_theme, t_group]), t_area, ts_out]))

## 5. Export (optional)

Region time series as a tidy table (damage, control, absolute and percent change per model / experiment / year), and the fetched cube as NetCDF. Pass `area="shelves"` to export the historical ice-shelf-only statistic; the default is the whole selected region.

In [ ]:
def region_timeseries(stat: str = "mean", area: str = "domain") -> pd.DataFrame:
    """Return regional damage and control differences as a tidy table."""
    # tidy DataFrame of the region statistic for all models / experiments
    series = _series(stat, area)
    ctrl = series.sel(experiment="ctrl", drop=True)
    out = xr.Dataset({
        "damage": series,
        "damage_ctrl": ctrl.broadcast_like(series),
        "delta": series - ctrl,
        "delta_pct": (series - ctrl) / ctrl.where(ctrl > EPS) * 100,
    }).drop_vars("time", errors="ignore")
    return out.to_dataframe().reset_index()

# Example (after fetching data):
df = region_timeseries("mean")
ipy_display(df.head())
df.to_csv("region_damage_timeseries.csv", index=False)
df
# STATE.ds.to_netcdf("damage_region.nc")